# Unsupervised Method Evaluation Against V3 Baseline

This notebook compares three candidate anomaly-detection methods on the same timestamp/floor/zone rows:

- `V3 statistical baseline`
- `LOF`
- `Optimized Isolation Forest`

There is no human ground-truth fault label in this dataset, so the evaluation uses proxy criteria that are appropriate for unsupervised anomaly detection: anomaly budget, persistence, cross-method agreement, operating-context awareness, explainability, and output completeness.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display, Markdown


def locate_project_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "Artificial Intelligence for Smart-Building Log Analysis",
        *cwd.parents,
        *[parent / "Artificial Intelligence for Smart-Building Log Analysis" for parent in cwd.parents],
    ]
    for candidate in candidates:
        if (candidate / "outputs").exists() and (candidate / "ml").exists():
            return candidate
    raise FileNotFoundError("Cannot locate project directory")


PROJECT_DIR = locate_project_dir()
OUTPUT_DIR = PROJECT_DIR / "outputs" / "floor6_zone1245_model_method_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IF_RESULTS_PATH = PROJECT_DIR / "outputs/isolation_forest_huaye/if_period_auto_results_full.csv.gz"
IF_EVENTS_PATH = PROJECT_DIR / "outputs/isolation_forest_huaye/if_period_auto_events.csv"
IF_SUMMARY_PATH = PROJECT_DIR / "outputs/isolation_forest_huaye/if_period_auto_modelling_summary.json"
LOF_SUMMARY_PATH = PROJECT_DIR / "outputs/floor6_zone1245_lof_modelling_gavin/lof_modelling_summary.json"
BASELINE_EVENTS_PATH = (
    PROJECT_DIR
    / "outputs/floor6_zone1245_2018_jul01_sep01_statistical_baseline_v3_15min_operation_aware"
    / "baseline_tables/11_baseline_anomaly_events.csv"
)

required_paths = [IF_RESULTS_PATH, IF_EVENTS_PATH, IF_SUMMARY_PATH, LOF_SUMMARY_PATH, BASELINE_EVENTS_PATH]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing required method outputs: {missing}")

print(f"Project directory: {PROJECT_DIR}")
print(f"Evaluation output directory: {OUTPUT_DIR}")


Project directory: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis
Evaluation output directory: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_model_method_evaluation


In [2]:
usecols = [
    "timestamp",
    "floor",
    "zone",
    "hour",
    "operating_period",
    "baseline_anomaly_label",
    "baseline_anomaly_count",
    "baseline_anomaly_methods",
    "baseline_anomaly_variables",
    "baseline_anomaly_directions",
    "lof_anomaly_label",
    "lof_anomaly_score",
    "period_if_anomaly_label",
    "period_if_auto_anomaly_label",
    "period_if_anomaly_score",
    "period_if_threshold_score",
]

aligned = pd.read_csv(IF_RESULTS_PATH, usecols=usecols, parse_dates=["timestamp"])
if aligned.duplicated(["timestamp", "floor", "zone"]).any():
    raise AssertionError("Aligned method table has duplicate timestamp/floor/zone rows")

with IF_SUMMARY_PATH.open(encoding="utf-8") as handle:
    if_summary = json.load(handle)
with LOF_SUMMARY_PATH.open(encoding="utf-8") as handle:
    lof_summary = json.load(handle)
baseline_events_native = pd.read_csv(BASELINE_EVENTS_PATH, parse_dates=["start_time", "end_time"])
if_events_native = pd.read_csv(IF_EVENTS_PATH, parse_dates=["start_time", "end_time"])

row_count = len(aligned)
display(pd.DataFrame([{
    "rows": row_count,
    "zones": aligned["zone"].nunique(),
    "start_time": aligned["timestamp"].min(),
    "end_time": aligned["timestamp"].max(),
    "operating_periods": ",".join(sorted(aligned["operating_period"].unique())),
}]).T.rename(columns={0: "value"}))


,value
rows,362880
zones,4
start_time,2018-07-01 00:00:00
end_time,2018-09-01 23:59:00
operating_periods,"core_work,evening_off,night_off,ramp_down,ramp..."


In [3]:
METHODS: dict[str, dict[str, Any]] = {
    "v3_baseline": {
        "method_name": "V3 statistical baseline",
        "label_col": "baseline_anomaly_label",
        "score_col": None,
        "method_family": "statistical_profile",
        "context_score": 1.00,
        "explainability_score": 1.00,
        "output_completeness_score": 0.95,
        "notes": "Strongest rule-level explanation; used as reference rather than ML ground truth.",
    },
    "lof": {
        "method_name": "LOF",
        "label_col": "lof_anomaly_label",
        "score_col": "lof_anomaly_score",
        "method_family": "unsupervised_ml",
        "context_score": 0.55,
        "explainability_score": 0.55,
        "output_completeness_score": 0.65,
        "notes": "Useful density-based comparator, but selected label still uses fixed contamination.",
    },
    "optimized_if": {
        "method_name": "Optimized Isolation Forest",
        "label_col": "period_if_anomaly_label",
        "score_col": "period_if_anomaly_score",
        "method_family": "unsupervised_ml",
        "context_score": 1.00,
        "explainability_score": 0.85,
        "output_completeness_score": 1.00,
        "notes": "Period-specific model with calibrated top-5% threshold and event aggregation.",
    },
}

EVENT_MIN_DURATION_MINUTES = int(if_summary.get("event_min_duration_minutes", 30))
EVENT_MAX_GAP_MINUTES = int(if_summary.get("event_max_gap_minutes", 2))
TARGET_ANOMALY_RATE_PCT = float(if_summary.get("calibrated_anomaly_rate", 0.05)) * 100

print(f"Target row-level anomaly budget: {TARGET_ANOMALY_RATE_PCT:.2f}%")
print(f"Common event rule: duration >= {EVENT_MIN_DURATION_MINUTES} minutes, max gap <= {EVENT_MAX_GAP_MINUTES} minutes")


Target row-level anomaly budget: 5.00%
Common event rule: duration >= 30 minutes, max gap <= 2 minutes


In [4]:
def jaccard(left: pd.Series, right: pd.Series) -> tuple[int, int, float]:
    left_mask = left.astype(bool)
    right_mask = right.astype(bool)
    overlap = int((left_mask & right_mask).sum())
    union = int((left_mask | right_mask).sum())
    return overlap, union, float(overlap / union) if union else 0.0


def build_row_events(frame: pd.DataFrame, label_col: str, method_key: str) -> pd.DataFrame:
    anomaly_rows = (
        frame.loc[frame[label_col].eq(1), ["timestamp", "zone", "operating_period", label_col]]
        .sort_values(["zone", "operating_period", "timestamp"])
        .copy()
    )
    if anomaly_rows.empty:
        return pd.DataFrame(
            columns=[
                "method_key",
                "event_id",
                "zone",
                "operating_period",
                "start_time",
                "end_time",
                "duration_minutes",
                "anomaly_points",
            ]
        )

    group_columns = ["zone", "operating_period"]
    gap_minutes = (
        anomaly_rows.groupby(group_columns, observed=True)["timestamp"]
        .diff()
        .dt.total_seconds()
        .div(60)
    )
    new_event = gap_minutes.isna() | gap_minutes.gt(EVENT_MAX_GAP_MINUTES)
    anomaly_rows["_event_group"] = new_event.groupby(
        [anomaly_rows["zone"], anomaly_rows["operating_period"]], observed=True
    ).cumsum()

    events = (
        anomaly_rows.groupby(group_columns + ["_event_group"], observed=True)
        .agg(
            start_time=("timestamp", "min"),
            end_time=("timestamp", "max"),
            anomaly_points=(label_col, "sum"),
        )
        .reset_index()
        .drop(columns="_event_group")
    )
    events["duration_minutes"] = (
        (events["end_time"] - events["start_time"]).dt.total_seconds().div(60).astype(int) + 1
    )
    events = events.loc[events["duration_minutes"].ge(EVENT_MIN_DURATION_MINUTES)].copy()
    events = events.sort_values(["duration_minutes", "anomaly_points"], ascending=[False, False]).reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1, dtype=int))
    events.insert(0, "method_key", method_key)
    return events


In [5]:
point_rows = []
event_frames = []

for method_key, config in METHODS.items():
    label_col = config["label_col"]
    score_col = config["score_col"]
    labels = aligned[label_col].astype(int)
    anomaly_count = int(labels.sum())
    anomaly_rate_pct = 100 * anomaly_count / row_count
    period_counts = aligned.loc[labels.eq(1), "operating_period"].value_counts(normalize=True)
    zone_counts = aligned.loc[labels.eq(1), "zone"].value_counts(normalize=True)

    events = build_row_events(aligned, label_col, method_key)
    event_frames.append(events)

    point_rows.append(
        {
            "method_key": method_key,
            "method_name": config["method_name"],
            "method_family": config["method_family"],
            "rows": row_count,
            "anomaly_count": anomaly_count,
            "anomaly_rate_pct": anomaly_rate_pct,
            "score_available": score_col is not None,
            "native_event_count": int(len(baseline_events_native)) if method_key == "v3_baseline" else int(len(if_events_native)) if method_key == "optimized_if" else np.nan,
            "common_event_count": int(len(events)),
            "common_event_anomaly_points": int(events["anomaly_points"].sum()) if not events.empty else 0,
            "persistent_point_share": (
                int(events["anomaly_points"].sum()) / anomaly_count if anomaly_count else 0.0
            ),
            "mean_common_event_duration": float(events["duration_minutes"].mean()) if not events.empty else 0.0,
            "median_common_event_duration": float(events["duration_minutes"].median()) if not events.empty else 0.0,
            "max_common_event_duration": int(events["duration_minutes"].max()) if not events.empty else 0,
            "period_coverage": int(aligned.loc[labels.eq(1), "operating_period"].nunique()),
            "zone_coverage": int(aligned.loc[labels.eq(1), "zone"].nunique()),
            "max_period_anomaly_share": float(period_counts.max()) if not period_counts.empty else 0.0,
            "max_zone_anomaly_share": float(zone_counts.max()) if not zone_counts.empty else 0.0,
            "mean_score_on_anomalies": float(aligned.loc[labels.eq(1), score_col].mean()) if score_col else np.nan,
            "notes": config["notes"],
        }
    )

point_metrics = pd.DataFrame(point_rows)
common_events = pd.concat(event_frames, ignore_index=True)

point_metrics.to_csv(OUTPUT_DIR / "01_method_point_and_event_metrics.csv", index=False)
common_events.to_csv(OUTPUT_DIR / "02_common_rule_events.csv", index=False)

display(point_metrics)
display(common_events.groupby("method_key", observed=True).head(5))


,method_key,method_name,method_family,rows,anomaly_count,anomaly_rate_pct,score_available,native_event_count,common_event_count,common_event_anomaly_points,persistent_point_share,mean_common_event_duration,median_common_event_duration,max_common_event_duration,period_coverage,zone_coverage,max_period_anomaly_share,max_zone_anomaly_share,mean_score_on_anomalies,notes
0,v3_baseline,V3 statistical baseline,statistical_profile,362880,46659,12.85797,False,475.0,397,46079,0.987569,116.078086,60.0,827,6,4,0.575795,0.279067,NaN,Strongest rule-level explanation; used as refe...
1,lof,LOF,unsupervised_ml,362880,18144,5.00000,True,NaN,138,9676,0.533289,70.195652,48.5,600,6,4,0.376653,0.304398,1.650192,"Useful density-based comparator, but selected ..."
2,optimized_if,Optimized Isolation Forest,unsupervised_ml,362880,18144,5.00000,True,134.0,134,13054,0.719466,97.552239,46.5,615,6,4,0.285714,0.434579,0.611530,Period-specific model with calibrated top-5% t...


,method_key,event_id,zone,operating_period,start_time,end_time,anomaly_points,duration_minutes
0,v3_baseline,1,4,weekend,2018-08-04 11:21:00,2018-08-05 01:07:00,827,827
1,v3_baseline,2,1,weekend,2018-08-04 10:34:00,2018-08-04 23:59:00,806,806
2,v3_baseline,3,5,weekend,2018-08-04 12:15:00,2018-08-04 23:59:00,705,705
3,v3_baseline,4,1,core_work,2018-07-30 08:00:00,2018-07-30 16:59:00,540,540
4,v3_baseline,5,1,core_work,2018-08-13 08:00:00,2018-08-13 16:59:00,540,540
397,lof,1,5,weekend,2018-08-26 00:00:00,2018-08-26 09:59:00,600,600
398,lof,2,4,weekend,2018-07-29 00:00:00,2018-07-29 06:59:00,420,420
399,lof,3,5,weekend,2018-08-04 23:07:00,2018-08-05 04:13:00,307,307
400,lof,4,5,core_work,2018-08-20 12:18:00,2018-08-20 16:59:00,282,282
401,lof,5,2,weekend,2018-07-22 09:00:00,2018-07-22 13:09:00,250,250


In [6]:
overlap_rows = []
method_items = list(METHODS.items())

for i, (left_key, left_config) in enumerate(method_items):
    for right_key, right_config in method_items[i + 1:]:
        overlap, union, score = jaccard(aligned[left_config["label_col"]], aligned[right_config["label_col"]])
        overlap_rows.append(
            {
                "left_method": left_config["method_name"],
                "right_method": right_config["method_name"],
                "left_key": left_key,
                "right_key": right_key,
                "overlap_count": overlap,
                "union_count": union,
                "jaccard_overlap": score,
            }
        )

pairwise_overlap = pd.DataFrame(overlap_rows)
pairwise_overlap.to_csv(OUTPUT_DIR / "03_pairwise_overlap.csv", index=False)

consensus_rows = []
for method_key, config in METHODS.items():
    relevant = pairwise_overlap.loc[
        pairwise_overlap["left_key"].eq(method_key) | pairwise_overlap["right_key"].eq(method_key)
    ]
    consensus_rows.append(
        {
            "method_key": method_key,
            "mean_pairwise_jaccard": float(relevant["jaccard_overlap"].mean()) if not relevant.empty else 0.0,
            "max_pairwise_jaccard": float(relevant["jaccard_overlap"].max()) if not relevant.empty else 0.0,
        }
    )
consensus_metrics = pd.DataFrame(consensus_rows)

all_three_mask = np.logical_and.reduce([
    aligned[config["label_col"]].astype(bool).to_numpy() for config in METHODS.values()
])
at_least_two_mask = sum(aligned[config["label_col"]].astype(bool) for config in METHODS.values()).ge(2)

display(pairwise_overlap)
display(pd.DataFrame([{
    "all_three_overlap_rows": int(all_three_mask.sum()),
    "at_least_two_methods_rows": int(at_least_two_mask.sum()),
    "at_least_two_methods_rate_pct": float(100 * at_least_two_mask.mean()),
}]))


,left_method,right_method,left_key,right_key,overlap_count,union_count,jaccard_overlap
0,V3 statistical baseline,LOF,v3_baseline,lof,2874,61929,0.046408
1,V3 statistical baseline,Optimized Isolation Forest,v3_baseline,optimized_if,6237,58566,0.106495
2,LOF,Optimized Isolation Forest,lof,optimized_if,1253,35035,0.035764


,all_three_overlap_rows,at_least_two_methods_rows,at_least_two_methods_rate_pct
0,225,9914,2.732033


In [7]:
evaluation = point_metrics.merge(consensus_metrics, on="method_key", how="left")

max_consensus = max(float(evaluation["mean_pairwise_jaccard"].max()), 1e-9)
evaluation["rate_score"] = (
    1 - (evaluation["anomaly_rate_pct"] - TARGET_ANOMALY_RATE_PCT).abs() / max(TARGET_ANOMALY_RATE_PCT * 2, 1)
).clip(lower=0, upper=1)
evaluation["persistence_score"] = evaluation["persistent_point_share"].clip(lower=0, upper=1)
evaluation["consensus_score"] = (evaluation["mean_pairwise_jaccard"] / max_consensus).clip(lower=0, upper=1)
evaluation["context_score"] = evaluation["method_key"].map({key: config["context_score"] for key, config in METHODS.items()})
evaluation["explainability_score"] = evaluation["method_key"].map({key: config["explainability_score"] for key, config in METHODS.items()})
evaluation["output_completeness_score"] = evaluation["method_key"].map({key: config["output_completeness_score"] for key, config in METHODS.items()})

WEIGHTS = {
    "rate_score": 0.25,
    "persistence_score": 0.15,
    "consensus_score": 0.20,
    "context_score": 0.20,
    "explainability_score": 0.15,
    "output_completeness_score": 0.05,
}

evaluation["overall_score"] = sum(evaluation[column] * weight for column, weight in WEIGHTS.items())
evaluation = evaluation.sort_values("overall_score", ascending=False).reset_index(drop=True)
evaluation["rank"] = np.arange(1, len(evaluation) + 1)

ranking_columns = [
    "rank",
    "method_name",
    "method_family",
    "overall_score",
    "anomaly_rate_pct",
    "common_event_count",
    "persistent_point_share",
    "mean_pairwise_jaccard",
    "rate_score",
    "persistence_score",
    "consensus_score",
    "context_score",
    "explainability_score",
    "output_completeness_score",
    "notes",
]

evaluation[ranking_columns].to_csv(OUTPUT_DIR / "04_method_evaluation_ranking.csv", index=False)
display(evaluation[ranking_columns])


,rank,method_name,method_family,overall_score,anomaly_rate_pct,common_event_count,persistent_point_share,mean_pairwise_jaccard,rate_score,persistence_score,consensus_score,context_score,explainability_score,output_completeness_score,notes
0,1,Optimized Isolation Forest,unsupervised_ml,0.921498,5.00000,134,0.719466,0.071130,1.000000,0.719466,0.930389,1.00,0.85,1.00,Period-specific model with calibrated top-5% t...
1,2,V3 statistical baseline,statistical_profile,0.799186,12.85797,397,0.987569,0.076452,0.214203,0.987569,1.000000,1.00,1.00,0.95,Strongest rule-level explanation; used as refe...
2,3,LOF,unsupervised_ml,0.662476,5.00000,138,0.533289,0.041086,1.000000,0.533289,0.537413,0.55,0.55,0.65,"Useful density-based comparator, but selected ..."


In [8]:
best = evaluation.iloc[0].to_dict()
second = evaluation.iloc[1].to_dict()

recommendation = {
    "selected_method": best["method_name"],
    "selected_method_key": best["method_key"],
    "overall_score": float(best["overall_score"]),
    "runner_up": second["method_name"],
    "runner_up_score": float(second["overall_score"]),
    "selection_basis": {
        "target_anomaly_rate_pct": TARGET_ANOMALY_RATE_PCT,
        "weights": WEIGHTS,
        "no_ground_truth_warning": "No human-confirmed fault label is available; this is a proxy evaluation for unsupervised candidate detection.",
    },
    "recommended_use": (
        "Use the selected method as the primary row-level candidate anomaly detector. "
        "Keep the v3 statistical baseline as the most interpretable diagnostic reference, and keep LOF as a secondary model comparison."
    ),
}

(OUTPUT_DIR / "method_selection_summary.json").write_text(
    json.dumps(recommendation, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

display(Markdown(
    f'''
## Recommended Method

**{recommendation['selected_method']}** is selected as the most suitable primary detection method under this proxy evaluation.

Why:

- It matches the calibrated anomaly budget of **{TARGET_ANOMALY_RATE_PCT:.1f}%**.
- It is operating-period aware.
- It produces persistent event outputs, not only point labels.
- It has stronger agreement with the v3 baseline than LOF in this run.
- It remains an unsupervised ML detector, while v3 remains the best rule-level explanatory reference.

Important limitation: this is **not** a ground-truth accuracy test, because no confirmed fault labels are available.
'''
))

display(pd.DataFrame([recommendation]).T.rename(columns={0: "value"}))



## Recommended Method

**Optimized Isolation Forest** is selected as the most suitable primary detection method under this proxy evaluation.

Why:

- It matches the calibrated anomaly budget of **5.0%**.
- It is operating-period aware.
- It produces persistent event outputs, not only point labels.
- It has stronger agreement with the v3 baseline than LOF in this run.
- It remains an unsupervised ML detector, while v3 remains the best rule-level explanatory reference.

Important limitation: this is **not** a ground-truth accuracy test, because no confirmed fault labels are available.


,value
selected_method,Optimized Isolation Forest
selected_method_key,optimized_if
overall_score,0.921498
runner_up,V3 statistical baseline
runner_up_score,0.799186
selection_basis,"{'target_anomaly_rate_pct': 5.0, 'weights': {'..."
recommended_use,Use the selected method as the primary row-lev...


In [9]:
print("Generated evaluation outputs:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.relative_to(PROJECT_DIR))


Generated evaluation outputs:
outputs\floor6_zone1245_model_method_evaluation\01_method_point_and_event_metrics.csv
outputs\floor6_zone1245_model_method_evaluation\02_common_rule_events.csv
outputs\floor6_zone1245_model_method_evaluation\03_pairwise_overlap.csv
outputs\floor6_zone1245_model_method_evaluation\04_method_evaluation_ranking.csv
outputs\floor6_zone1245_model_method_evaluation\05_budget_sensitivity_summary.csv
outputs\floor6_zone1245_model_method_evaluation\method_selection_summary.json
